# Vector Databases & ANN Indexes

Companion notebook for the [Vector Databases wiki page](https://ml-viz.vercel.app/wiki/vector-databases).

We build two indexes from scratch over synthetic embeddings: an exact **Flat** index (brute-force
cosine) and an approximate **IVF** index (k-means coarse quantizer + `nprobe`). Then we measure the
core trade-off — **recall@k vs. speedup** as we turn the `nprobe` knob. Pure NumPy, no network.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

rng = np.random.default_rng(42)

## 1 — Synthetic embeddings

We draw clustered, L2-normalized vectors (clustered data is what makes IVF effective — real
embeddings have topic structure). Normalizing once means **cosine ranking == dot-product ranking**,
so we can use the fast dot product everywhere.

In [ ]:
def normalize(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)

N, d, n_clusters = 50_000, 64, 50
centers = rng.normal(size=(n_clusters, d))
labels = rng.integers(0, n_clusters, size=N)
X = normalize(centers[labels] + 0.35 * rng.normal(size=(N, d)))   # corpus
Q = normalize(centers[rng.integers(0, n_clusters, 200)] + 0.35 * rng.normal(size=(200, d)))  # queries
print(f"corpus X: {X.shape},  queries Q: {Q.shape}")

## 2 — Flat (exact) index — the ground truth

For unit vectors, cosine similarity is just the dot product, so top-$k$ is the $k$ largest entries of
$X q^\top$. Exact and simple, but it scans all $N$ vectors.

In [ ]:
class FlatIndex:
    def __init__(self, X):
        self.X = X
    def search(self, q, k=10):
        sims = self.X @ q                         # O(N*d)
        idx = np.argpartition(-sims, k)[:k]
        return idx[np.argsort(-sims[idx])]

flat = FlatIndex(X)
truth = np.array([flat.search(q, k=10) for q in Q])    # ground-truth neighbours
print("top-10 for query 0:", truth[0])

## 3 — IVF index from scratch

**Build:** k-means the corpus into `nlist` cells; each vector is assigned to its nearest centroid.
**Search:** find the `nprobe` nearest centroids to the query, then brute-force *only* the vectors in
those cells. `nprobe` is the recall/latency knob.

In [ ]:
def kmeans(X, k, iters=15, seed=0):
    r = np.random.default_rng(seed)
    C = X[r.choice(len(X), k, replace=False)]
    for _ in range(iters):
        assign = np.argmax(X @ C.T, axis=1)            # nearest centroid (cosine)
        for j in range(k):
            pts = X[assign == j]
            if len(pts):
                C[j] = pts.mean(0)
        C = normalize(C)
    return C, np.argmax(X @ C.T, axis=1)

class IVFIndex:
    def __init__(self, X, nlist=200):
        self.X = X
        self.centroids, assign = kmeans(X, nlist)
        self.cells = [np.where(assign == j)[0] for j in range(nlist)]
    def search(self, q, k=10, nprobe=8):
        probe = np.argsort(-(self.centroids @ q))[:nprobe]      # nearest cells
        cand = np.concatenate([self.cells[j] for j in probe])
        sims = self.X[cand] @ q
        top = cand[np.argsort(-sims)[:k]]
        return top, len(cand)

ivf = IVFIndex(X, nlist=200)
approx, n_scanned = ivf.search(Q[0], k=10, nprobe=8)
print(f"IVF scanned {n_scanned} of {N} vectors ({100*n_scanned/N:.1f}%) for one query")

## 4 — Recall@k vs. nprobe (the trade-off curve)

**Recall@k** = fraction of the true top-$k$ that the approximate search actually returned. As
`nprobe` rises, recall climbs toward 1.0 but we scan more vectors (less speedup). This curve is the
whole story of ANN tuning.

In [ ]:
def recall_at_k(approx, truth):
    return np.mean([len(set(a) & set(t)) / len(t) for a, t in zip(approx, truth)])

nprobes = [1, 2, 4, 8, 16, 32, 64, 128, 200]
recalls, fracs = [], []
for np_ in nprobes:
    res = [ivf.search(q, k=10, nprobe=np_) for q in Q]
    approx = np.array([r[0] for r in res])
    recalls.append(recall_at_k(approx, truth))
    fracs.append(np.mean([r[1] for r in res]) / N)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(nprobes, recalls, 'o-', color='#2dd4bf', label='recall@10')
ax1.set_xlabel('nprobe (cells searched)'); ax1.set_ylabel('recall@10', color='#2dd4bf')
ax1.set_xscale('log', base=2); ax1.set_ylim(0, 1.02); ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(nprobes, [1/f for f in fracs], 's--', color='#fb7185', label='speedup vs flat')
ax2.set_ylabel('speedup vs flat (×)', color='#fb7185')
ax1.set_title('IVF: recall rises and speedup falls as nprobe grows')
plt.tight_layout(); plt.show()

for np_, r, f in zip(nprobes, recalls, fracs):
    print(f"nprobe={np_:3d}   recall@10={r:.3f}   scanned={100*f:5.1f}%   speedup≈{1/f:5.1f}x")

## ✏️ Your turn

**Exercise.** Implement `min_nprobe_for_recall(ivf, Q, truth, target, k=10)` that returns the
**smallest** `nprobe` (search the sorted list `[1,2,4,8,16,32,64,128,200]`) whose measured recall@k
is at least `target`. This is exactly the tuning question in production: *what is the cheapest
setting that still meets my recall SLA?*

In [ ]:
def measured_recall(ivf, Q, truth, nprobe, k=10):
    approx = np.array([ivf.search(q, k=k, nprobe=nprobe)[0] for q in Q])
    return recall_at_k(approx, truth)

def min_nprobe_for_recall(ivf, Q, truth, target, k=10):
    grid = [1, 2, 4, 8, 16, 32, 64, 128, 200]
    # TODO(you): return the smallest nprobe in `grid` with measured_recall >= target,
    #            or grid[-1] if none reach the target
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
np50 = min_nprobe_for_recall(ivf, Q, truth, target=0.50)
np90 = min_nprobe_for_recall(ivf, Q, truth, target=0.90)
assert measured_recall(ivf, Q, truth, np90) >= 0.90
assert np90 >= np50, "a higher recall target needs at least as many probes"
print(f"✓ recall>=0.50 needs nprobe={np50};  recall>=0.90 needs nprobe={np90}")

<details>
<summary>Solution</summary>

```python
def min_nprobe_for_recall(ivf, Q, truth, target, k=10):
    grid = [1, 2, 4, 8, 16, 32, 64, 128, 200]
    for np_ in grid:
        if measured_recall(ivf, Q, truth, np_, k) >= target:
            return np_
    return grid[-1]
```

Recall is monotonic in `nprobe` (more cells can only add candidates), so the first grid value that
clears the target is the cheapest. Picking it is the recall-vs-latency tuning every vector DB
deployment does — usually targeting something like recall@10 ≥ 0.95 at the lowest latency.

</details>